In [ ]:
from collections import Counter, defaultdict
from itertools import chain, combinations, permutations
from math import comb
from pathlib import Path

import dedupe
import numpy as np
import polars as pl
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import train_test_split

In [ ]:
DATASET_PATH = Path("datasets/features_dataset_20260707.parquet")

In [ ]:
RANDOM_SEED = 42

# Chargement du jeu de données

In [ ]:
df_features = pl.read_parquet(DATASET_PATH)

In [ ]:
df_features.shape

In [ ]:
df_train = df_features.filter(pl.col("split") == "train")
df_test = df_features.filter(pl.col("split") == "test")
print(df_train.shape, df_test.shape)

In [ ]:
df_train.columns

In [ ]:
df_train.describe()

# Transformation du jeu de données au format dedupe

In [ ]:
FEATURES_NAMES_FROM_DATASET = [
    "nom",
    "description",
    "acteur_type_id",
    "source_id",
    "adresse",
    "adresse_complement",
    "code_postal",
    "ville",
    "url",
    "email",
    "telephone",
    "nom_commercial",
    "nom_officiel",
    "siren",
    "siret",
    "naf_principal",
    "horaires_osm",
    "horaires_description",
    "horaires_osm",
    "public_accueilli",
    "reprise",
    "exclusivite_de_reprisereparation",
    "reprise",
    "uniquement_sur_rdv",
    "consignes_dacces",
    "action_principale_id",
    "lieu_prestation",
    "code_commune_insee",
    "epci_id",
    "action_reparer",
    "action_acheter",
    "action_revendre",
    "action_donner",
    "action_louer",
    "action_mettreenlocation",
    "action_emprunter",
    "action_preter",
    "action_echanger",
    "action_trier",
    "action_rapporter",
]

In [ ]:
def build_entities(df_features: pl.DataFrame, feature_names: list[str]) -> dict:
    """
    Construit un dictionnaire {id_entité: {feature: valeur, ...}} unique
    à partir des colonnes _i / _j de chaque paire.
    """
    entities = {}
    for row in df_features.iter_rows(named=True):
        for suffix, id_col in (
            ("_i", "identifiant_unique_i"),
            ("_j", "identifiant_unique_j"),
        ):
            eid = row[id_col]
            if eid not in entities:
                entity = {}
                for feature_name in feature_names:
                    value = row[f"{feature_name}{suffix}"]
                    if isinstance(value, int) or isinstance(value, float):
                        value = str(value)
                    entity[feature_name] = value
                entity["location"] = (
                    row[f"latitude{suffix}"],
                    row[f"longitude{suffix}"],
                )
                entities[eid] = entity

    return entities

In [ ]:
entities_dict = build_entities(df_features, feature_names=FEATURES_NAMES_FROM_DATASET)

In [ ]:
entities_dict

In [ ]:
def build_ground_truth_clusters(df_pairs: pl.DataFrame) -> dict:
    """
    Construit un mapping id_entité -> cluster vrai, COMPLET sur toutes les
    entités présentes dans `pairs_df`.

    Les entités absentes d'un cluster (utilisées uniquement comme
    exemples négatifs) sont traitées comme des clusters singletons — elles
    n'ont pas de duplicat connu — avec un identifiant synthétique unique
    par entité, plutôt que d'être exclues de la vérité terrain. Sans ça,
    ces entités seraient silencieusement ignorées lors de l'évaluation
    (un faux positif les concernant ne serait jamais compté) et toujours
    reléguées dans train_sub lors du split train/dev.
    """

    df_identifiant_unique_to_cluster_id = pl.concat(
        [
            df_pairs.filter(pl.col("label")).select(
                pl.col("identifiant_unique_i").alias("identifiant_unique"), "cluster_id"
            ),
            df_pairs.filter(pl.col("label")).select(
                pl.col("identifiant_unique_j").alias("identifiant_unique"), "cluster_id"
            ),
        ]
    ).unique("identifiant_unique")
    id_to_cluster = dict(
        zip(
            df_identifiant_unique_to_cluster_id["identifiant_unique"].to_list(),
            df_identifiant_unique_to_cluster_id["cluster_id"].to_list(),
        )
    )
    all_ids = set(df_pairs["identifiant_unique_i"].to_list()) | set(
        df_pairs["identifiant_unique_j"].to_list()
    )
    for eid in all_ids:
        id_to_cluster.setdefault(eid, f"__singleton_true__{eid}")
    return id_to_cluster

In [ ]:
# Vérité terrain complète (id -> cluster_id), construite une seule fois
id_to_cluster_full = build_ground_truth_clusters(df_features)

In [ ]:
id_to_cluster_full

# Train / dev split

In [ ]:
pairs_train = df_train
pairs_test = df_test

In [ ]:
def split_train_dev(
    df_train_features: pl.DataFrame,
    dev_ratio: float = 0.2,
    seed: int = RANDOM_SEED,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    train_cluster_ids, dev_cluster_ids = train_test_split(
        df_train_features.select("cluster_id").unique("cluster_id"),
        test_size=dev_ratio,
        random_state=seed,
    )

    df_train_sub = df_train_features.filter(
        pl.col("cluster_id").is_in(train_cluster_ids.get_column("cluster_id").to_list())
    )

    df_dev = df_train_features.filter(
        pl.col("cluster_id").is_in(dev_cluster_ids.get_column("cluster_id").to_list())
    )

    return df_train_sub, df_dev

In [ ]:
df_train_sub, df_dev = split_train_dev(df_train)

In [ ]:
df_train_sub.shape, df_dev.shape

In [ ]:
assert (
    len(
        set(
            df_train_sub.select(pl.col("cluster_id").unique())
            .get_column("cluster_id")
            .to_list()
        )
        & set(
            df_dev.select(pl.col("cluster_id").unique())
            .get_column("cluster_id")
            .to_list()
        )
    )
    == 0
)

# Configuration des variables

In [ ]:
variables = [
    dedupe.variables.String(name="nom", field="nom"),
    dedupe.variables.Text("description", has_missing=True),
    dedupe.variables.Categorical(
        "acteur_type_id",
        categories=["1", "2", "3", "4", "5", "7", "8", "9", "10", "11", "12"],
    ),
    dedupe.variables.String(name="adresse", field="adresse", has_missing=True),
    dedupe.variables.String(
        name="adresse_complement", field="adresse_complement", has_missing=True
    ),
    dedupe.variables.Exact(name="code_postal", field="code_postal", has_missing=True),
    dedupe.variables.String(name="ville", field="ville", has_missing=True),
    dedupe.variables.Exact("url", has_missing=True),
    dedupe.variables.Exact("email", has_missing=True),
    dedupe.variables.Exact("telephone", has_missing=True),
    dedupe.variables.String(
        name="nom_commercial", field="nom_commercial", has_missing=True
    ),
    dedupe.variables.String(
        name="nom_officiel", field="nom_officiel", has_missing=True
    ),
    dedupe.variables.Exact("siren", has_missing=True),
    dedupe.variables.Exact("siret", has_missing=True),
    dedupe.variables.Exact("naf_principal", has_missing=True),
    dedupe.variables.Text(name="horaires_osm", field="horaires_osm", has_missing=True),
    dedupe.variables.Text(
        name="horaires_description", field="horaires_description", has_missing=True
    ),
    dedupe.variables.Categorical(
        "public_accueilli",
        categories=[
            "Aucun",
            "Particuliers",
            "Particuliers et professionnels",
            "Professionnels",
        ],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        name="reprise",
        field="reprise",
        categories=["1 pour 0", "1 pour 1"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        name="exclusivite_de_reprisereparation",
        field="exclusivite_de_reprisereparation",
        categories=["True", "False"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        "uniquement_sur_rdv", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Text("consignes_dacces", has_missing=True),
    dedupe.variables.Categorical(
        "action_principale_id",
        categories=["7", "8", "5", "6", "1", "9", "3", "2", "12", "11", "4"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        "lieu_prestation",
        categories=["A_DOMICILE", "SUR_PLACE", "SUR_PLACE_OU_A_DOMICILE"],
        has_missing=True,
    ),
    dedupe.variables.LatLong("location"),
    dedupe.variables.Exact("code_commune_insee", has_missing=True),
    dedupe.variables.Exact("epci_id", has_missing=True),
    dedupe.variables.Categorical(
        "action_reparer", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_acheter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_revendre", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_donner", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_louer", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_mettreenlocation", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_emprunter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_preter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_echanger", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_trier", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_rapporter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Interaction(
        "nom",
        "nom_commercial",
        "nom_officiel",
    ),
    dedupe.variables.Interaction(
        "adresse",
        "adresse_complement",
        "code_postal",
        "ville",
    ),
    dedupe.variables.Interaction("horaires_osm", "horaires_description"),
    dedupe.variables.Interaction("reprise", "exclusivite_de_reprisereparation"),
]

# Entrainement du modèle

In [ ]:
def train_deduper(
    df_train_sub: pl.DataFrame, entities: dict, dedupe_variables_definition: list
) -> dedupe.Dedupe:
    """
    Entraîne un objet dedupe.Dedupe à partir des paires labellisées de
    `df_train_sub`, sans passer par l'apprentissage actif interactif.

    entities contient tous les acteurs dans un dictionnaire format dedupe.
    """
    deduper = dedupe.Dedupe(dedupe_variables_definition)

    train_ids = set(df_train_sub["identifiant_unique_i"].to_list()) | set(
        df_train_sub["identifiant_unique_j"].to_list()
    )
    train_entities = {i: entities[i] for i in train_ids}

    # Nécessaire pour initialiser le "sampler" interne de dedupe, même si
    # on ne se sert pas de l'échantillonnage actif ensuite.
    deduper.prepare_training(
        train_entities, sample_size=max(1000, len(df_train_sub) * 2)
    )

    labeled_pairs = {"match": [], "distinct": []}
    for row in df_train_sub.iter_rows(named=True):
        pair = (
            entities[row["identifiant_unique_i"]],
            entities[row["identifiant_unique_j"]],
        )
        labeled_pairs["match" if row["label"] else "distinct"].append(pair)
    deduper.mark_pairs(labeled_pairs)
    deduper.train()
    deduper.cleanup_training()

    return deduper

In [ ]:
deduper = train_deduper(df_train_sub, entities_dict, variables)

In [ ]:
deduper

# Sélection du seuil

In [ ]:
def partition_to_dict(clustered, all_ids: set) -> dict:
    """
    Convertit la sortie de `deduper.partition()` en dict {id: cluster_label}.
    Les entités que dedupe n'a rattachées à aucun cluster (cas limite selon
    versions) sont explicitement placées dans un cluster singleton.
    """
    id_to_cluster = {}
    for cluster_idx, (ids, _scores) in enumerate(clustered):
        for eid in ids:
            id_to_cluster[eid] = f"c_{cluster_idx}"
    for eid in all_ids:
        id_to_cluster.setdefault(eid, f"singleton_{eid}")
    return id_to_cluster


def predict_pairs_from_clusters(
    df_pairs: pl.DataFrame, id_to_cluster: dict
) -> list[bool]:
    """Prédit le label 'duplicat' d'une paire par co-appartenance au même cluster."""
    return [
        id_to_cluster.get(row["identifiant_unique_i"])
        == id_to_cluster.get(row["identifiant_unique_j"])
        for row in df_pairs.iter_rows(named=True)
    ]

In [ ]:
def pairwise_metrics_from_clusters_2(
    df_pairs: pl.DataFrame, pred_clusters: list
) -> dict:
    pred_pairs = [
        tuple(sorted(e))
        for cluster in pred_clusters
        for e in combinations(
            cluster[0],
            2,
        )
    ]

    tp = 0
    fp = 0
    for pair in pred_pairs:
        df_search = df_pairs.filter(
            pl.col("label")
            & (pl.col("identifiant_unique_i") == pair[0])
            & (pl.col("identifiant_unique_j") == pair[1])
        )
        if len(df_search) == 0:
            fp += 1
        else:
            tp += 1

    fn = 0
    for row in df_pairs.filter(pl.col("label")).iter_rows(named=True):
        if (row["identifiant_unique_i"], row["identifiant_unique_j"]) not in pred_pairs:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

In [ ]:
def pairwise_metrics_from_clusters(true_clusters: dict, pred_clusters: dict) -> dict:
    """
    Precision / recall / F1 au niveau paire, calculés de façon EXHAUSTIVE
    sur toutes les paires possibles entre les entités considérées (et pas
    seulement sur les paires présentes dans `pairs_df`).

    En effet, `deduper.partition()` regroupe les entités en clusters : deux
    entités peuvent se retrouver dans le même cluster prédit sans avoir
    jamais formé une paire annotée dans le dataframe de base. Ignorer ces
    paires "nouvelles" revient à ne jamais compter certains faux positifs
    (ni certains faux négatifs pour des paires vraies non couvertes par
    l'échantillon), ce qui biaise la métrique à la hausse.

    On évite d'énumérer explicitement les O(n²) paires en s'appuyant sur
    une formule combinatoire à partir des tailles de clusters (vrais,
    prédits, et intersections vrai×prédit) : pour un groupe de taille n,
    il y a C(n,2) paires internes.

    `true_clusters` et `pred_clusters` doivent couvrir exactement le même
    ensemble d'identifiants (typiquement : toutes les entités du split
    évalué pour lesquelles la vérité terrain est connue).
    """
    ids = list(true_clusters.keys())
    assert set(ids) == set(
        pred_clusters.keys()
    ), "true_clusters et pred_clusters doivent couvrir les mêmes ids"

    # nb de paires prédites comme duplicats = somme des C(taille,2) par cluster prédit
    pred_sizes = Counter(pred_clusters[i] for i in ids)
    predicted_positive = sum(comb(n, 2) for n in pred_sizes.values())

    # nb de paires réellement duplicats = somme des C(taille,2) par cluster vrai
    true_sizes = Counter(true_clusters[i] for i in ids)
    actual_positive = sum(comb(n, 2) for n in true_sizes.values())

    # vrais positifs = paires qui sont ensemble à la fois dans le cluster prédit ET le cluster vrai
    joint_sizes = Counter((pred_clusters[i], true_clusters[i]) for i in ids)
    tp = sum(comb(n, 2) for n in joint_sizes.values())

    fp = predicted_positive - tp
    fn = actual_positive - tp

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

In [ ]:
def fbeta(precision: float, recall: float, beta: float = 0.5) -> float:
    """F-bêta : beta < 1 favorise la précision par rapport au rappel."""
    if precision == 0 and recall == 0:
        return 0.0
    b2 = beta**2
    return (1 + b2) * precision * recall / (b2 * precision + recall)


def b_cubed(true_clusters: dict, pred_clusters: dict) -> dict:
    """
    Métrique B-cubed (précision / rappel / F1), standard pour comparer un
    clustering prédit à un clustering de référence, entité par entité.
    """
    ids = list(true_clusters.keys())

    pred_groups = defaultdict(set)
    true_groups = defaultdict(set)
    for i in ids:
        pred_groups[pred_clusters[i]].add(i)
        true_groups[true_clusters[i]].add(i)

    precisions, recalls = [], []
    for i in ids:
        c_pred = pred_groups[pred_clusters[i]]
        c_true = true_groups[true_clusters[i]]
        inter = len(c_pred & c_true)
        precisions.append(inter / len(c_pred))
        recalls.append(inter / len(c_true))

    p = float(np.mean(precisions))
    r = float(np.mean(recalls))
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"b3_precision": p, "b3_recall": r, "b3_f1": f1}

In [ ]:
def select_best_threshold(
    deduper: dedupe.Dedupe,
    dev_entities: dict,
    id_to_cluster_dev: dict,
    df_pairs_dev: pl.DataFrame,
    thresholds=np.arange(0.10, 1.00, 0.05),
    min_recall: float = 0.3,
) -> tuple[float, dict]:
    """
    Teste une grille de seuils sur le jeu de dev et choisit celui qui :
      - maximise la précision, parmi les seuils qui respectent un rappel
        minimal (`min_recall`) ;
      - à défaut (aucun seuil ne respecte le rappel minimal), maximise le
        F-bêta (bêta=0.5, qui pondère davantage la précision).

    La précision/rappel est calculée de façon EXHAUSTIVE sur toutes les
    entités du dev ayant une vérité terrain connue (via `clusters_df`),
    et non sur le seul sous-échantillon de paires annotées : cela évite
    d'ignorer les faux positifs "inventés" par le clustering de dedupe.

    NB : on relance `partition()` pour chaque seuil (simplification). Sur
    un très gros volume, on pourrait factoriser le blocking/scoring et ne
    faire varier que l'étape de clustering.
    """
    eval_ids = set(id_to_cluster_dev.keys())

    results = []
    for t in thresholds:
        # On demande à dedupe de clusteriser et on obtient une liste de liste d'ids regroupés
        clustered = deduper.partition(dev_entities, float(t))

        # On crée un dict entity_id -> cluster_id + singleton
        id_to_cluster_pred = partition_to_dict(clustered, set(dev_entities.keys()))
        # On s'assure qu'il n'y ait que des ids du jeu d'évaluation
        pred_restr = {i: id_to_cluster_pred[i] for i in eval_ids}

        # On calcule les métriques en comparant
        metrics = pairwise_metrics_from_clusters(id_to_cluster_dev, pred_restr)
        metrics_2 = pairwise_metrics_from_clusters_2(df_pairs_dev, clustered)
        results.append((float(t), metrics))
        print(
            f"  seuil={t:.2f}  precision={metrics['precision']:.3f}  "
            f"recall={metrics['recall']:.3f}  f1={metrics['f1']:.3f}"
        )
        print(
            f"precision2={metrics_2['precision']:.3f}  "
            f"recall2={metrics_2['recall']:.3f}  f12={metrics_2['f1']:.3f}"
        )
        print("-----")

    # 1) parmi les seuils respectant le rappel minimal, on prend le plus précis
    eligibles = [(t, m) for t, m in results if m["recall"] >= min_recall]
    if eligibles:
        return max(eligibles, key=lambda x: x[1]["precision"])

    # 2) sinon, repli sur le F-bêta (favorise quand même la précision)
    return max(
        results, key=lambda x: fbeta(x[1]["precision"], x[1]["recall"], beta=0.5)
    )

In [ ]:
dev_ids = set(df_dev["identifiant_unique_i"].to_list()) | set(
    df_dev["identifiant_unique_j"].to_list()
)
dev_entities = {i: entities_dict[i] for i in dev_ids}
id_to_cluster_dev = {i: id_to_cluster_full[i] for i in dev_ids}

best_threshold, best_dev_metrics = select_best_threshold(
    deduper, dev_entities, id_to_cluster_dev, df_dev
)

In [ ]:
len(dev_entities), len(id_to_cluster_dev)

In [ ]:
best_threshold, best_dev_metrics

# Application des règles métiers

In [ ]:
def _cluster_has_violation(ids: tuple, entities: dict) -> bool:
    """Vrai si au moins une paire du cluster viole une règle métier."""

    for a, b in combinations(ids, 2):
        source_id_a = entities[a].get("source_id")
        source_id_b = entities[b].get("source_id")
        if all(e is not None for e in (source_id_a, source_id_b)) and (
            source_id_a == source_id_b
        ):
            print("Violation")
            return True

        acteur_type_id_a = entities[a].get("acteur_type_id")
        acteur_type_id_b = entities[b].get("acteur_type_id")
        if all(e is not None for e in (acteur_type_id_a, acteur_type_id_b)) and (
            acteur_type_id_a != acteur_type_id_b
        ):
            print("Violation")
            return True

    return False


def apply_business_rules(
    clustered, entities: dict, strategy: str = "strict"
) -> list[tuple]:
    """
    Post-traite la sortie de `deduper.partition()` pour faire respecter
    des règles métier strictes :
      - deux entités ayant le même `source_id` ne peuvent jamais être
        dans le même cluster ;
      - deux entités ayant un `acteur_type` différent ne peuvent jamais
        être dans le même cluster.

    `entities` : dict {id: {feature: valeur, ...}} devant contenir au
    minimum les clés "source_id" et "acteur_type" pour chaque entité.

    Retourne une liste de tuples (ids, scores), au même format que la
    sortie de `deduper.partition()`.
    """

    final_clusters = []
    num_clusters = 0
    num_clusters_with_violation = 0
    for ids, scores in clustered:

        if len(ids) < 2:
            final_clusters.append((ids, scores))
            continue
        num_clusters += 1

        if _cluster_has_violation(ids, entities):
            num_clusters_with_violation += 1
            # cluster écarté : chaque entité redevient un singleton
            for eid, score in zip(ids, scores):
                final_clusters.append(((eid,), (score,)))
        else:
            final_clusters.append((ids, scores))
        continue
    print("Num clusters before :", num_clusters)
    print("Num clusters with violation :", num_clusters_with_violation)
    return final_clusters


def partition_with_rules(
    deduper: dedupe.Dedupe,
    data: dict,
    threshold: float,
    entities: dict,
    strategy: str = "strict",
) -> list[tuple]:
    """
    Partitionne avec dedupe puis applique les règles métier de
    post-filtrage. À utiliser systématiquement à la place d'un appel brut
    à `deduper.partition(...)`, y compris lors de la sélection du seuil,
    pour que le seuil choisi reflète le comportement réel du pipeline
    (règles métier comprises).
    """
    clustered = deduper.partition(data, threshold)
    return apply_business_rules(clustered, entities, strategy=strategy)

# Evaluation sur le test set

In [ ]:
df_test.group_by("cluster_id").agg(
    pl.col("identifiant_unique_i")
    .list.concat(pl.col("identifiant_unique_j"))
    .explode()
    .alias("ids")
).with_columns(pl.col("ids").list.n_unique().alias("taille_cluster")).group_by(
    "taille_cluster"
).agg(
    pl.len().alias("nombre_cluster")
).sort(
    "taille_cluster"
)

In [ ]:
test_ids = set(df_test["identifiant_unique_i"].to_list()) | set(
    df_test["identifiant_unique_j"].to_list()
)
test_entities = {i: entities_dict[i] for i in test_ids}

clustered_test = partition_with_rules(
    deduper, test_entities, best_threshold, test_entities
)
id_to_cluster_pred = partition_to_dict(clustered_test, test_ids)

id_to_cluster_true = {i: id_to_cluster_full[i] for i in test_ids}
id_to_cluster_pred_restr = {i: id_to_cluster_pred[i] for i in test_ids}

In [ ]:
cluster_test_with_cluster_id = []
for c_id, cluster in enumerate(clustered_test):
    if len(cluster[0]) > 1:
        for actor_id, score in zip(cluster[0], cluster[1]):
            cluster_test_with_cluster_id.append(
                {
                    "cluster_id": c_id,
                    "actor_id": actor_id,
                    "score_confiance": score,
                }
            )

In [ ]:
cluster_test_with_cluster_id

In [ ]:
pl.DataFrame(cluster_test_with_cluster_id).write_csv(
    DATASET_PATH.parent / "test_clusters_pred_20260709.csv"
)

In [ ]:
df_test.group_by("cluster_id").agg(
    pl.col("identifiant_unique_i")
    .list.concat(pl.col("identifiant_unique_j"))
    .explode()
    .alias("ids"),
    pl.col("label").first(),
).with_columns(pl.col("ids").list.unique().list.sort()).sort("cluster_id").explode(
    "ids"
).write_csv(
    DATASET_PATH.parent / "test_clusters_20260709.csv"
)

## Metriques pairwise

In [ ]:
pairwise_res = pairwise_metrics_from_clusters(
    id_to_cluster_true, id_to_cluster_pred_restr
)

In [ ]:
pairwise_res

## Métrique au niveau cluster

In [ ]:
b3_res = b_cubed(id_to_cluster_true, id_to_cluster_pred_restr)
print(f"[TEST] Cluster (B-cubed) : {b3_res}")

## Exploration des résultats

In [ ]:
clustered_test

In [ ]:
sum(len(e[0]) > 1 for e in clustered_test)

In [ ]:
Counter([len(e[0]) for e in clustered_test])

### Transformation en paires

In [ ]:
pred_pairs_test = [
    tuple(sorted(e))
    for cluster in clustered_test
    for e in combinations(
        cluster[0],
        2,
    )
]

pred_pairs_test_dicts = [
    {"identifiant_unique_i": e[0], "identifiant_unique_j": e[1]}
    for e in pred_pairs_test
]

In [ ]:
pred_pairs_test_dicts

In [ ]:
df_pred_pairs = pl.DataFrame(pred_pairs_test_dicts)

### Paires en commun

In [ ]:
df_test_common = df_test.join(
    df_pred_pairs,
    on=["identifiant_unique_i", "identifiant_unique_j"],
    how="inner",
    validate="1:1",
)

In [ ]:
with pl.Config(set_fmt_str_lengths=200):
    print(
        df_test.join(
            df_pred_pairs,
            on=["identifiant_unique_i", "identifiant_unique_j"],
            how="inner",
            validate="1:1",
        )
    )

In [ ]:
df_test_common.select(
    "identifiant_unique_i",
    "identifiant_unique_j",
    "cluster_id",
    "label",
    "nom_i",
    "nom_j",
).to_dicts()

### Paires côté preds

In [ ]:
df_pred_pairs.join(
    df_test, on=["identifiant_unique_i", "identifiant_unique_j"], how="anti"
)

In [ ]:
df_features.filter(
    (pl.col("identifiant_unique_i") == "aliapur_21141")
    | (pl.col("identifiant_unique_j") == "aliapur_21141")
)